# ゼロから作るショアのアルゴリズム 第3回: 制御剰余乗算

[第1回](ja/401_shor_scratch_adder.ipynb)では通常の加算器を、[第2回](ja/402_shor_scratch_modulo_adder.ipynb)では剰余加算 $a+b \bmod N$ を作りました。今回は**制御剰余乗算**を実装します。固定された古典定数 $a$ が与えられたとき、論文の式(11)をそのまま目標にします。

$$
\ket{c; x, 0} \longrightarrow
\begin{cases}
\ket{c; x,\ a x \bmod N} & (\ket c = \ket 1) \\
\ket{c; x,\ x} & (\ket c = \ket 0)
\end{cases}
$$

つまり、$x$ に定数 $a$ を掛けて $N$ で割った余りを計算しますが、これは追加の制御量子ビット $c$ が $1$ のときだけ行われ、$c=0$ のときは結果レジスタに $x$ がそのままコピーされます。これはまさに、次回作る**べき剰余**を構築するために繰り返し使う演算です: $a^j \bmod N$ は、$j$ の各ビットで制御された $a, a^2, a^4, a^8, \ldots$ による制御乗算の連鎖です。

$c=0$ のときになぜ $\ket 0$ ではなく $\ket x$ なのかは、このノートブックの最後で説明します。

参考文献: V. Vedral, A. Barenco, A. Ekert, ["Quantum Networks for Elementary Arithmetic Operations"](https://arxiv.org/abs/quant-ph/9511018) (1995), Figure 5

## アイデア: 乗算は条件付き足し算の繰り返し

2進数の筆算による乗算とは: $x$ の各ビット $x_i$ について、$x_i=1$ ならば $a \cdot 2^i$ を合計に加える、というものです。論文も同じことを言っています。

> Function $f_{a,N}(x) = ax \bmod N$ can be implemented by repeated conditional additions (modulo $N$).

この処理を(最後に一度だけでなく)**毎回 $N$ で割った余りを取りながら**行うことで、すべての中間値を範囲内に収めます。これは重要です。なぜなら、加算器のレジスタは $N$ 未満の値しか保持できるように設計されていないからです。

したがって回路は次のようになります: $i = 0, \ldots, n-1$ について、第2回の剰余加算器を使って*定数* $(a \cdot 2^i) \bmod N$ を結果レジスタ $b$ に加えます — ただし、重ね合わせの中で**乗算器自身の制御量子ビット $c$** と $x_i$ の**両方**が $1$ になっている枝だけに対して行います。

問題は「2つの量子ビットが両方とも1のときだけ」をどう実装するかです。第2回のステップ4で見た手口が、ここでもそのまま効きます。

In [ ]:
!pip install git+https://github.com/blueqat/blueqatSDK

## Fig.5 の手口: 制御するのは加算器ではなく「定数」のほう

素直に考えると、第2回で作った剰余加算器の**全ゲートに制御を2つ足す**ことになります。しかしそうすると、第2回で最大2制御(トフォリ)だったゲートが4制御になり、多重制御ゲートの分解が必要になって回路が急激に膨れます。

論文はそうしていません。

> To account for this fact at the $i$th modular addition stage the first register is loaded with the value $2^i a$ if $\ket{c, x_i} = \ket{1,1}$ and with value 0 otherwise. This is done by applying the Toffoli gate to the control qubits $\ket c$ and $\ket{x_i}$ and the appropriate target qubit in the register; the gate is applied each time value "1" appears in the binary form of the number $2^i a$.

つまり、**加算器には一切手を触れない**。代わりに、足す数を置く作業レジスタ(以下 `const`)への書き込みのほうを条件付きにする。$(c, x_i) = (1,1)$ のときだけ $2^i a$ を、そうでなければ **0** をロードする。

なぜこれで済むのか。**$0$ を足す剰余加算は何もしないから**です。$b < N$ なら $(0 + b) \bmod N = b$。加算器そのものは無条件に走らせてよく、入力が $0$ なら勝手に恒等変換になってくれます。

第2回のステップ4($N$ レジスタを条件付きに $0$ にする)と、まったく同じ発想です。論文はこの手口を Fig.4 と Fig.5 の2箇所で使っています。**演算そのものを条件付きにするのではなく、入力を条件付きにして演算が自明になるようにする。**

そしてロードを元に戻すのも簡単です。

> Resetting the register to its initial state is done by applying the same sequence of the Toffoli gates again (the order of the gates is irrelevant as they act on different target qubits).

トフォリは自己逆元なので、**同じ列をもう一度流すだけ**でアンロードになります。

この方針の帰結として、**このノートブックの回路にはトフォリ(`CCX`)を超えるゲートが一度も現れません**。多重制御ゲートの分解も、そのための補助量子ビット(アンシラ)も不要です。

## 第2回の部品をそのまま持ってくる

剰余加算器は第2回のものをそのまま使います(論文 Fig.4 どおりの、$N$ レジスタを条件付きにゼロ化する版です)。

In [2]:
from blueqat import Circuit

def carry(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[a, b, c_next]
    circ.cx[a, b]
    circ.ccx[c, b, c_next]
    return circ

def carry_inv(c, a, b, c_next):
    circ = Circuit()
    circ.ccx[c, b, c_next]
    circ.cx[a, b]
    circ.ccx[a, b, c_next]
    return circ

def sum_gate(c, a, b):
    circ = Circuit()
    circ.cx[a, b]
    circ.cx[c, b]
    return circ

def add_circuit(c, a, b, n):
    """b := a + b, using carry register c. b must have n+1 qubits."""
    circ = Circuit()
    for i in range(n - 1):
        circ += carry(c[i], a[i], b[i], c[i + 1])
    circ += carry(c[n - 1], a[n - 1], b[n - 1], b[n])
    circ.cx[a[n - 1], b[n - 1]]
    circ += sum_gate(c[n - 1], a[n - 1], b[n - 1])
    for i in range(n - 2, -1, -1):
        circ += carry_inv(c[i], a[i], b[i], c[i + 1])
        circ += sum_gate(c[i], a[i], b[i])
    return circ

def sub_circuit(c, a, b, n):
    """b := b - a: the adder's gate list, reversed."""
    add_circ = add_circuit(c, a, b, n)
    reversed_circ = Circuit()
    reversed_circ.ops = list(reversed(add_circ.ops))
    return reversed_circ

def set_to_zero_if(t, N, val_N, n):
    """Reset the N register to 0 when t == 0. Self-inverse."""
    circ = Circuit()
    circ.x[t]
    for j in range(n):
        if (val_N >> j) & 1:
            circ.cx[t, N[j]]
    circ.x[t]
    return circ

def modulo_add_circuit(c, a, b, N, t, n, val_N):
    """b := (a + b) mod N, on the given registers. Paper Fig.4."""
    circ = Circuit()
    circ += add_circuit(c, a, b, n)          # 1. b += a
    circ += sub_circuit(c, N, b, n)          # 2. b -= N
    circ.cx[b[n], t]                         # 3. overflow -> t
    gate = set_to_zero_if(t, N, val_N, n)
    circ += gate                             # 4. blank N when t == 0 ...
    circ += add_circuit(c, N, b, n)          #    ... plain adder ...
    circ += gate                             #    ... restore N
    circ += sub_circuit(c, a, b, n)          # 5. clean up
    circ.x[b[n]]
    circ.cx[b[n], t]
    circ.x[b[n]]
    circ += add_circuit(c, a, b, n)
    return circ

## 制御乗算器

量子ビットは全部で $5n+3$ 個です。

- `c[0..n-1]`: 桁上げレジスタ
- `const[0..n-1]`: 作業用。各段で $(a \cdot 2^i) \bmod N$ を置く
- `b[0..n]`: 結果レジスタ($n+1$ 幅)
- `N[0..n-1]`: 法
- `x[0..n-1]`: 掛けられる数
- `ctrl`: 乗算器の制御量子ビット $c$
- `t`: 剰余加算器のあふれフラグ

$x$ の各ビット $i$ について:

1. `ctrl` と $x_i$ で制御したトフォリで、古典定数 $(a \cdot 2^i) \bmod N$ を `const` にロードする
2. **制御なしの**剰余加算器を走らせる
3. 同じトフォリの列をもう一度流して `const` を $\ket 0$ に戻す

古典定数を回路構築時に計算できるのは、$a$、$N$、$i$ がすべて既知の定数であり、重ね合わせに依存しないからです。

最後に、式(11)の「$c=0$ なら $x$ をコピー」を付けます(論文の Fig.5 のいちばん右にあるトフォリの列)。

In [3]:
def build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n):
    c     = list(range(n))                     # carry register
    const = list(range(n, 2 * n))              # scratch: holds (a * 2**i) mod N
    b     = list(range(2 * n, 3 * n + 1))      # result register (n+1 wide)
    N     = list(range(3 * n + 1, 4 * n + 1))  # modulus
    x     = list(range(4 * n + 1, 5 * n + 1))  # multiplicand
    ctrl  = 5 * n + 1                          # multiplier's control qubit
    t     = 5 * n + 2                          # modulo-adder's overflow flag
    circ = Circuit(5 * n + 3)                  # no ancilla needed

    if ctrl_value:
        circ.x[ctrl]
    for i in range(n):
        if (val_x >> i) & 1:
            circ.x[x[i]]
        if (val_N >> i) & 1:
            circ.x[N[i]]

    # The adder carries no extra controls, and is identical at every stage,
    # so it can be built once outside the loop and reused.
    adder = modulo_add_circuit(c, const, b, N, t, n, val_N)

    for i in range(n):
        const_val = (mult_a * (1 << i)) % val_N
        load = Circuit()
        for j in range(n):
            if (const_val >> j) & 1:
                load.ccx[ctrl, x[i], const[j]]   # <- the only controlled part
        circ += load      # load 2^i * a if (c, x_i) == (1, 1), else 0
        circ += adder     # b := (const + b) mod N
        circ += load      # Toffolis are self-inverse -> unload

    # eq.(11): if ctrl == 0, copy x into the result register
    circ.x[ctrl]
    for i in range(n):
        circ.ccx[ctrl, x[i], b[i]]
    circ.x[ctrl]

    return circ, b

In [4]:
def run_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n=None, quiet=False):
    """a, x, N, ctrl can be any values you like -- n (register width) is inferred
    from N and x automatically if not given explicitly."""
    if n is None:
        n = max(val_N, val_x, 1).bit_length()
    circuit, b = build_controlled_multiplier(mult_a, val_x, val_N, ctrl_value, n)
    total = circuit.n_qubits
    if not quiet:
        print(f"  [n={n}, qubits={total}, gates={len(circuit.ops)}]")
    state = circuit.m[:].run(shots=1, backend="statevector").most_common(1)[0][0]
    bits = [state[total - 1 - q] for q in b]
    return int("".join(reversed(bits)), 2)

for a, x, N, ctrl in [(2, 1, 3, 1), (2, 2, 3, 1), (2, 1, 3, 0), (2, 2, 3, 0)]:
    result = run_controlled_multiplier(a, x, N, ctrl)
    expected = (a * x) % N if ctrl else x        # <- eq.(11)
    print(f"ctrl={ctrl}: {a}*{x} mod {N} = {result}  (expected {expected}, "
          f"{'OK' if result == expected else 'MISMATCH'})")

  [n=2, qubits=13, gates=176]


ctrl=1: 2*1 mod 3 = 2  (expected 2, OK)
  [n=2, qubits=13, gates=176]


ctrl=1: 2*2 mod 3 = 1  (expected 1, OK)
  [n=2, qubits=13, gates=175]


ctrl=0: 2*1 mod 3 = 1  (expected 1, OK)
  [n=2, qubits=13, gates=175]


ctrl=0: 2*2 mod 3 = 2  (expected 2, OK)


`ctrl=1` では掛け算が実行され、`ctrl=0` では結果レジスタに $x$ がそのまま出てきます。式(11)のとおりです。

## 網羅テスト

$N=7$ について、$x$ を全通り、`ctrl` を両方で試します。回路は線形なので、計算基底のすべての入力で正しければ、その重ね合わせでも正しく動きます。

In [5]:
bad = 0
for ctrl in (0, 1):
    for x in range(7):
        result = run_controlled_multiplier(3, x, 7, ctrl, n=3, quiet=True)
        expected = (3 * x) % 7 if ctrl else x
        if result != expected:
            bad += 1
            print(f"MISMATCH ctrl={ctrl} x={x}: {result} != {expected}")
print(f"14 cases checked, {bad} mismatches")

14 cases checked, 0 mismatches


## 規模についての補足

この回路は普通のPythonコードだけで組み立てられる(量子的な重ね合わせを必要としない)ので、実行する前に量子ビット数とゲート数を正確に表示できます。$N=15$ — ショアのアルゴリズムの古典的な最初の例 — まで含めて見てみます。

In [6]:
for a, x, N, n in [(2, 2, 3, 2), (3, 5, 7, 3), (7, 13, 15, 4)]:
    circ, _ = build_controlled_multiplier(a, x, N, 1, n)
    print(f"N={N:2d} (n={n}): {circ.n_qubits:2d} qubits, {len(circ.ops):4d} gates")

N= 3 (n=2): 13 qubits,  176 gates
N= 7 (n=3): 18 qubits,  395 gates
N=15 (n=4): 23 qubits,  702 gates


量子ビット数は $5n+3$ と $n$ に対して**線形**にしか増えません。ところが、この1ショットを古典的にシミュレートするコストはそうはいきません。筆者の環境での実測は次のとおりでした。

| | 量子ビット | ゲート数 | 1ショットの時間 |
|---|---|---|---|
| $N=3$ | 13 | 176 | 0.3秒 |
| $N=7$ | 18 | 395 | 5.4秒 |
| $N=15$ | 23 | 702 | **2047秒** |

状態ベクトルのシミュレーションでは、1ゲートあたりのコストが振幅の数、つまり $2^{(\text{量子ビット数})}$ に比例します。18量子ビットから23量子ビットで振幅は $2^5 = 32$ 倍、そこにゲート数1.8倍が掛かって素朴には**60倍**。ところが実測は**380倍**でした(小さい回路はPythonのゲート単位のオーバーヘッドが支配的で、大きい回路はメモリ帯域が律速になるため、大きくなるほど見積もりより悪化します)。

これは blueqat が遅いという話ではありません。回路を組み立てるのは $n$ の多項式時間で終わるのに、それを古典的に真似しようとすると指数で殴られる — **素因数分解が古典的に難しいと考えられている理由**を、手元で少しだけ体感できる部分です。実機であれば、回路を実行するコストは*シミュレータの*制約とはまったく関係がありません。

$N=15$ を実際に走らせたい場合は、次のセルのコメントを外してください(約34分かかります)。

In [7]:
# result = run_controlled_multiplier(7, 13, 15, 1, n=4)   # 7 * 13 mod 15 = 1
# print(result)

## なぜ $c=0$ のとき $\ket 0$ ではなく $\ket x$ なのか

最後に式(11)の後半を回収します。

この乗算器がやっているのは $U_{a,N}\ket{x, 0} \to \ket{x, ax \bmod N}$ で、**入力 $x$ が残ったまま**です。ショアのアルゴリズムで本当に欲しいのは、同じレジスタが $\ket x \to \ket{ax \bmod N}$ と書き換わる写像です。論文はそれを、あらかじめ古典計算で $a^{-1} \bmod N$ を求めておいて(ユークリッドの互除法で効率よく計算できます)、

1. $U_{a,N}$ で $\ket{x, 0} \to \ket{x,\ ax}$
2. 2つのレジスタを**交換**して $\ket{ax,\ x}$
3. $U_{a^{-1},N}$ の**逆**を掛けて $\ket{ax,\ 0}$

という3段で実現します。3で消えるのは、$U_{a^{-1},N}\ket{ax, 0} = \ket{ax,\ a^{-1}ax} = \ket{ax,\ x}$ だからです。

ここで $c = 0$ の場合を考えます。制御が $0$ なら全体が恒等変換、つまり $\ket{x,0}$ のまま出てきてほしい。

- もし $c=0$ で結果レジスタが $\ket 0$ のままだと、2の交換で $\ket{0,\ x}$ になってしまう。$x$ が向こう側に行ってしまい、戻ってこない
- $c=0$ で $\ket{x,x}$ になっていれば、交換しても $\ket{x,x}$ のまま。3の逆演算が2つ目のレジスタを $\ket 0$ に戻して、めでたく $\ket{x,0}$

コピーがあるおかげで、**交換そのものは制御なしでよくなる**わけです。式(11)の一見不思議な後半は、次回の構成のための伏線でした。

## 次回予告

制御乗算ができたので、次の部品は[べき剰余](ja/404_shor_scratch_modular_exponentiation.ipynb)です — この回路を核として、位相推定レジスタの各ビットで制御しながら繰り返し二乗していきます。上で説明したレジスタ交換と $a^{-1} \bmod N$ による逆演算が、そこで実際に登場します。

これがショアのアルゴリズムの量子部分が実際に実行する演算であり、このシリーズ全体を[400_shor.ipynb](ja/400_shor.ipynb)の位数発見の理論と最終的に結びつける部品です。